# D-Fire — Error Analysis

Finds image-level false positives / false negatives and builds failure
galleries, so we can see *which* scenes the model gets wrong (sunsets, clouds,
distant flames) rather than only reading aggregate metrics.

Set `WEIGHTS` to a trained checkpoint and `DATA_ROOT` to the dataset root.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = None  # auto-detect if None
WEIGHTS = str(REPO_ROOT / "outputs" / "weights" / "yolo_main" / "weights" / "best.pt")
SPLIT = "val"

from src.utils import resolve_data_root
from src.inference.detector import WildfireDetector
data_root = resolve_data_root(DATA_ROOT)
detector = WildfireDetector.from_config(REPO_ROOT / "configs" / "inference.yaml", mode="balanced", weights=WEIGHTS)

In [ ]:
# Collect image-level false positives (predicts hazard on a 'none' image)
# and false negatives (misses a real hazard), then show galleries.
from src.data.dfire import iter_split, parse_label_file, image_category
from src.visualization.plot_predictions import _read_rgb, make_comparison_grid

false_pos, false_neg = [], []
for img_path, lbl_path in iter_split(data_root / SPLIT / "images"):
    gt_boxes, _ = parse_label_file(lbl_path)
    gt_cat = image_category(gt_boxes)
    dets = detector.predict(str(img_path))
    alert = detector.alert_state(dets)
    pred_has = alert.value != "none"
    gt_has = gt_cat != "none"
    if pred_has and not gt_has:
        false_pos.append((detector.annotate(_read_rgb(img_path), dets), f"FP: {img_path.name}"))
    elif gt_has and not pred_has:
        false_neg.append((_read_rgb(img_path), f"FN ({gt_cat}): {img_path.name}"))
    if len(false_pos) >= 6 and len(false_neg) >= 6:
        break

print(f"Collected {len(false_pos)} FP and {len(false_neg)} FN examples")
if false_pos:
    make_comparison_grid(false_pos[:6], REPO_ROOT / "outputs" / "figures" / "false_positives.png", suptitle="False positives")
if false_neg:
    make_comparison_grid(false_neg[:6], REPO_ROOT / "outputs" / "figures" / "false_negatives.png", suptitle="False negatives")

In [ ]:
# Where do the two error types come from? Summarize the categories we missed.
from collections import Counter

fn_categories = Counter(caption.split("(")[1].split(")")[0] for _, caption in false_neg)
print("False negatives by ground-truth category:", dict(fn_categories))
print(f"Total FP: {len(false_pos)} | Total FN: {len(false_neg)}")